In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# Carreguem
df = pd.read_csv('../data/processed/df_features.csv')
y  = pd.read_csv('../data/processed/y.csv').squeeze()

# Separem train i test
train = df.iloc[:len(y)]
test  = df.iloc[len(y):]

print(f"Train: {train.shape}")
print(f"Test:  {test.shape}")
print(f"Target: {y.shape}")

Train: (1458, 246)
Test:  (1459, 246)
Target: (1458,)


In [3]:
# Escalem — Ridge necessita features a la mateixa escala
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train)
test_scaled  = scaler.transform(test)

# Cross-validation amb 5 folds
ridge = Ridge(alpha=10)
scores = cross_val_score(ridge, train_scaled, y, 
                         cv=5, scoring='neg_root_mean_squared_error')

print(f"Ridge CV log-RMSE: {-scores.mean():.4f} ± {scores.std():.4f}")

Ridge CV log-RMSE: 0.1215 ± 0.0064


In [4]:
xgb_model = xgb.XGBRegressor(n_estimators=1000, learning_rate=0.05, 
                               random_state=42)
scores_xgb = cross_val_score(xgb_model, train, y,
                              cv=5, scoring='neg_root_mean_squared_error')

print(f"XGBoost CV log-RMSE: {-scores_xgb.mean():.4f} ± {scores_xgb.std():.4f}")

XGBoost CV log-RMSE: 0.1221 ± 0.0045


In [5]:
from sklearn.model_selection import GridSearchCV

alphas = [0.1, 1, 5, 10, 50, 100, 300, 500]
ridge_grid = GridSearchCV(Ridge(), {'alpha': alphas}, 
                          cv=5, scoring='neg_root_mean_squared_error')
ridge_grid.fit(train_scaled, y)

print(f"Ridge millor alpha: {ridge_grid.best_params_}")
print(f"Ridge millor CV log-RMSE: {-ridge_grid.best_score_:.4f}")

Ridge millor alpha: {'alpha': 300}
Ridge millor CV log-RMSE: 0.1166


In [6]:
xgb_tuned = xgb.XGBRegressor(
    n_estimators=2000,
    learning_rate=0.01,
    max_depth=4,
    min_child_weight=2,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

scores_xgb_tuned = cross_val_score(xgb_tuned, train, y,
                                    cv=5, scoring='neg_root_mean_squared_error')

print(f"XGBoost tuned CV log-RMSE: {-scores_xgb_tuned.mean():.4f} ± {scores_xgb_tuned.std():.4f}")

XGBoost tuned CV log-RMSE: 0.1137 ± 0.0060


In [7]:
# Ridge — entrenem amb totes les dades de train
ridge_final = Ridge(alpha=300)
ridge_final.fit(train_scaled, y)
pred_ridge_train = ridge_final.predict(train_scaled)
pred_ridge_test  = ridge_final.predict(test_scaled)

# XGBoost — entrenem amb totes les dades de train
xgb_final = xgb.XGBRegressor(
    n_estimators=2000, learning_rate=0.01,
    max_depth=4, min_child_weight=2,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42
)
xgb_final.fit(train, y)
pred_xgb_train = xgb_final.predict(train)
pred_xgb_test  = xgb_final.predict(test)

print("Models entrenats!")

Models entrenats!


In [8]:
# Blend — mitjana ponderada de les dues prediccions
# Donem més pes a XGBoost perquè té millor CV score
blend_train = 0.4 * pred_ridge_train + 0.6 * pred_xgb_train
blend_test  = 0.4 * pred_ridge_test  + 0.6 * pred_xgb_test

# Avaluem el blend al train
from sklearn.metrics import mean_squared_error
rmse_blend = np.sqrt(mean_squared_error(y, blend_train))
print(f"Blend log-RMSE train: {rmse_blend:.4f}")

# Comparem els tres

print(f"Blend train: {rmse_blend:.4f}")

Blend log-RMSE train: 0.0618
Blend train: 0.0618


In [9]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)
blend_scores = []

for train_idx, val_idx in kf.split(train):
    # Split
    X_tr, X_val = train.iloc[train_idx], train.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # Ridge
    X_tr_sc  = scaler.fit_transform(X_tr)
    X_val_sc = scaler.transform(X_val)
    ridge_final.fit(X_tr_sc, y_tr)
    p_ridge = ridge_final.predict(X_val_sc)
    
    # XGBoost
    xgb_final.fit(X_tr, y_tr)
    p_xgb = xgb_final.predict(X_val)
    
    # Blend
    p_blend = 0.4 * p_ridge + 0.6 * p_xgb
    score = np.sqrt(mean_squared_error(y_val, p_blend))
    blend_scores.append(score)


print(f"\nResum:")
print(f"Ridge    CV: {-ridge_grid.best_score_:.4f}")
print(f"XGBoost  CV: {-scores_xgb_tuned.mean():.4f} ± {scores_xgb_tuned.std():.4f}")
print(f"Blend    CV: {np.mean(blend_scores):.4f} ± {np.std(blend_scores):.4f}")


Resum:
Ridge    CV: 0.1166
XGBoost  CV: 0.1137 ± 0.0060
Blend    CV: 0.1100 ± 0.0073


In [10]:
# Desfer el log1p per obtenir preus reals
submission = pd.read_csv('../data/raw/sample_submission.csv')
submission['SalePrice'] = np.expm1(blend_test)

submission.to_csv('../submissions/submission_02_blend.csv', index=False)
print("Submission guardada!")
print(submission.head())

Submission guardada!
     Id      SalePrice
0  1461  119882.982127
1  1462  162749.138380
2  1463  179876.196851
3  1464  193880.286194
4  1465  187158.700369


In [11]:
import lightgbm as lgb

lgb_model = lgb.LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.01,
    max_depth=4,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)

scores_lgb = cross_val_score(lgb_model, train, y,
                              cv=5, scoring='neg_root_mean_squared_error')

print(f"LightGBM CV log-RMSE: {-scores_lgb.mean():.4f} ± {scores_lgb.std():.4f}")

LightGBM CV log-RMSE: 0.1206 ± 0.0050


In [15]:
lgb_fast = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    random_state=42,
    verbose=-1
)

scores_lgb_fast = cross_val_score(lgb_fast, train, y,
                                   cv=5, scoring='neg_root_mean_squared_error')
print(f"LightGBM fast CV log-RMSE: {-scores_lgb_fast.mean():.4f} ± {scores_lgb_fast.std():.4f}")

LightGBM fast CV log-RMSE: 0.1219 ± 0.0054


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
blend_scores = []

for train_idx, val_idx in kf.split(train):
    X_tr, X_val = train.iloc[train_idx], train.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # Ridge
    X_tr_sc  = scaler.fit_transform(X_tr)
    X_val_sc = scaler.transform(X_val)
    ridge_final.fit(X_tr_sc, y_tr)
    p_ridge = ridge_final.predict(X_val_sc)
    
    # XGBoost
    xgb_final.fit(X_tr, y_tr)
    p_xgb = xgb_final.predict(X_val)
    
    # LightGBM
    lgb_fast.fit(X_tr, y_tr)
    p_lgb = lgb_fast.predict(X_val)
    
    # Blend 3 models
    p_blend = 0.3 * p_ridge + 0.5 * p_xgb + 0.2 * p_lgb
    score = np.sqrt(mean_squared_error(y_val, p_blend))
    blend_scores.append(score)

print(f"Blend 3 models CV log-RMSE: {np.mean(blend_scores):.4f} ± {np.std(blend_scores):.4f}")

Blend 2  CV: 0.1109 ± 0.0079
Blend 3 models CV log-RMSE: 0.1109 ± 0.0079
